In [22]:
"""
SV callset preprocessing notebook
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from pysam import VariantFile

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as sv_utils

importlib.reload(vcfp)
importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

preprocessed_savedir = Path('saved/preprocessed/')
preprocessed_savedir.mkdir(parents=True, exist_ok=True)

samples = {}
for sample_name in ['CHM13', 'NA12878', 'NA12878_30x']:
    samples.update(sv_utils.create_config_dict(sample_name, is_reference=False))
samples.update(sv_utils.create_config_dict('gnomad', is_reference=True))

samples

# callsets = {
#     'gnomad': {
#         'source': 'gnomad',
#         'path': Path('data/gnomad.v4.1.sv.sites.vcf.gz'),
#         'params': caller_params.gnomad_params
#     },
#     'gridss': {
#         'source': 'gridss',
#         'path': Path('data/test_gridss_output.NA12878.vcf'),
#         'params': caller_params.gridss_params
#     },
#     'manta_candidate': {
#         'source': 'manta_candidate',
#         'path': Path('data/candidateSV.vcf.gz'),
#         'params': caller_params.manta_candidate_params
#     },
#     'manta_diploid': {
#         'source': 'manta_diploid',
#         'path': Path('data/diploidSV.vcf.gz'),
#         'params': caller_params.manta_diploid_params
#     }
# }

to_process_list = ['gnomad', 'gridss', 'manta_candidate', 'manta_diploid']

In [23]:
samples['gnomad']

{'sample_dir': PosixPath('data/gnomad'),
 'is_reference': True,
 'callsets': {'gnomad': {'source': 'gnomad',
   'path': PosixPath('data/gnomad/gnomad.vcf.gz'),
   'params': {'base_cols': ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL'],
    'filter_cols': ['PASS'],
    'info_cols_mandatory': ['EVIDENCE',
     'SVTYPE',
     'CHR2',
     'POS2',
     'END',
     'END2',
     'CPX_INTERVALS',
     'SVLEN'],
    'info_cols_pred': [],
    'parse_format': False,
    'format_cols': [],
    'is_reference': True,
    'svlen_source': 'svlen',
    'supported_svtypes': ['DEL', 'DUP', 'INV', 'INS'],
    'method_cipos': {'SR': 10, 'PE': 100, 'RD': 500}}}}}

In [20]:
for sample, sample_config in samples.items():
    if sample_config['is_reference']:
        # Check if the reference sample contains only 1 "callset"
        if len(sample_config['callsets']) > 1:
            raise ValueError(f'Reference sample {sample} contains multiple callsets: {sample_config["callsets"].keys()}')
        # Preprocess the reference sample
        if sample_config['callsets'][sample_config['callsets'].keys()[0]]['source'] == 'gnomad':
            gnomad_vcf = VariantFile(sample_config['callsets'][sample]['path'])
            gnomad_df, gnomad_meta = vcfp.parse_vcf_to_df(vcf_file=gnomad_vcf, caller_params=sample_config['callsets'][sample]['params'])
            gnomad_df = caller_specific.gnomad_preprocess(gnomad_df, caller_params=sample_config['callsets'][sample]['params'], verbose=True)
            gnomad_bnd_df = caller_specific.gnomad_event_to_bnd(gnomad_df)
            gnomad_bp_df = bnds.bnd_to_bp(gnomad_bnd_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
            if not (sample_config['sample_dir'] / 'svensemble2').exists():
                (sample_config['sample_dir'] / 'svensemble2').mkdir(parents=True, exist_ok=True)
            gnomad_bp_df.to_csv(sample_config['sample_dir'] / 'svensemble2' / 'gnomad_bps.csv')
            del gnomad_df, gnomad_meta, gnomad_bnd_df, gnomad_bp_df
        else:
            raise ValueError(f'Unknown reference sample: {sample}')
        continue

    if 'gridss' in sample_config['callsets'].keys():
        gridss_vcf = VariantFile(sample_config['callsets']['gridss']['path'])
        gridss_df, gridss_meta = vcfp.parse_vcf_to_df(vcf_file=gridss_vcf, caller_params=sample_config['callsets']['gridss']['params'])
        gridss_df = caller_specific.gridss_preprocess(gridss_df, caller_params=sample_config['callsets']['gridss']['params'], verbose=True)
        gridss_bp_df = bnds.bnd_to_bp(gridss_df)
        if not (sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        gridss_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'gridss' / 'svensemble2' / 'gridss_bps.csv')
        del gridss_df, gridss_meta, gridss_bp_df
    if 'manta_diploid' in to_process_list:
        manta_diploid_vcf = VariantFile(sample_config['callsets']['manta_diploid']['path'])
        manta_diploid_df, manta_diploid_meta = vcfp.parse_vcf_to_df(vcf_file=manta_diploid_vcf, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_bnd_only_df = manta_diploid_df[manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_evt_only_df = manta_diploid_df[~manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_bnd_only_df, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_evt_only_df, caller_params=sample_config['callsets']['manta_diploid']['params'])
        manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
        manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
        manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
        if not (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        manta_diploid_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2' / 'manta_diploid_bps.csv')
        del manta_diploid_df, manta_diploid_meta, manta_bnd_only_df, manta_evt_only_df, manta_evt_bnd_df, manta_diploid_bp_df
    if 'manta_candidate' in sample_config['callsets'].keys():
        manta_candidate_vcf = VariantFile(sample_config['callsets']['manta_candidate']['path'])
        manta_candidate_df, manta_candidate_meta = vcfp.parse_vcf_to_df(vcf_file=manta_candidate_vcf, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_bnd_only_df = manta_candidate_df[manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_candidate_evt_only_df = manta_candidate_df[~manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
        manta_candidate_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_candidate_bnd_only_df, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_candidate_evt_only_df, caller_params=sample_config['callsets']['manta_candidate']['params'])
        manta_candidate_evt_bnd_df = evts.event_to_bnd(manta_candidate_evt_only_df)
        manta_candidate_df = pd.concat([manta_candidate_bnd_only_df, manta_candidate_evt_bnd_df], axis=0)
        manta_candidate_bp_df = bnds.bnd_to_bp(manta_candidate_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
        if not (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').exists():
            (sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2').mkdir(parents=True, exist_ok=True)
        manta_candidate_bp_df.to_csv(sample_config['sample_dir'] / 'sv_calling' / 'manta' / 'svensemble2' / 'manta_candidate_bps.csv')
        del manta_candidate_df, manta_candidate_meta, manta_candidate_bnd_only_df, manta_candidate_evt_only_df, manta_candidate_evt_bnd_df, manta_candidate_bp_df


Dropping ['REF', 'ASC', 'BEALN', 'BEID', 'BEIDH', 'BEIDL', 'BENAMES', 'BPNAMES', 'CIRPOS', 'HOMLEN', 'HOMSEQ', 'IHOMPOS', 'INSRM', 'INSRMRC', 'INSRMRO', 'INSRMRT', 'INSTAXID', 'RSI', 'SC', 'SI'] from meta as it is not present in specified info cols
Dropping all format fields
Preprocessing GRIDSS callset...
Splitting CIPOS values...
Dropping INFO_CIPOS column...
Returning processed callset...
{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 15, 'chr17': 16, 'chr18': 17, 'chr19': 18, 'chr20': 19, 'chr21': 20, 'chr22': 21, 'chrX': 22, 'chrY': 23, 'chrM': 24}
Dropping ['END', 'CIGAR', 'HOMSEQ', 'SVINSLEN', 'SVINSSEQ', 'LEFT_SVINSSEQ', 'RIGHT_SVINSSEQ', 'MATE_BND_DEPTH', 'JUNCTION_QUAL'] from meta as it is not present in specified info cols
Dropping ['Ploidy', 'MaxDepth', 'MaxMQ0Frac', 'NoPairSupport', 'MinQUAL', 'SampleFT', 'MinGQ', 'HomRef'] from meta as

[E::idx_find_and_load] Could not retrieve index file for 'data/gnomad/gnomad.vcf.gz'


Preprocessing gnomAD callset...
SV types currently supported: ['DEL', 'DUP', 'INV', 'INS']
Using default evidence CIPOS: {'SR': 10, 'PE': 100, 'RD': 500}
Using default viable SV types: ['DEL', 'DUP', 'INV', 'INS']
Filtering to specified SV types: ['DEL', 'DUP', 'INV', 'INS']
Setting confidence intervals based on evidence type
Setting EVENT field to match variant ID
Returning processed callset
{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 15, 'chr17': 16, 'chr18': 17, 'chr19': 18, 'chr20': 19, 'chr21': 20, 'chr22': 21, 'chrX': 22, 'chrY': 23, 'chrM': 24}


In [65]:
if 'gridss' in to_process_list:
    gridss_vcf = VariantFile(callsets['gridss']['path'])
    gridss_df, gridss_meta = vcfp.parse_vcf_to_df(vcf_file=gridss_vcf, caller_params=callsets['gridss']['params'])
    gridss_df = caller_specific.gridss_preprocess(gridss_df, caller_params=callsets['gridss']['params'], verbose=True)
    gridss_bp_df = bnds.bnd_to_bp(gridss_df)
    gridss_bp_df.to_csv(preprocessed_savedir / 'gridss_bps.csv')
    del gridss_df, gridss_meta, gridss_bp_df


Dropping ['REF', 'ASC', 'BEALN', 'BEID', 'BEIDH', 'BEIDL', 'BENAMES', 'BPNAMES', 'CIRPOS', 'HOMLEN', 'HOMSEQ', 'IHOMPOS', 'INSRM', 'INSRMRC', 'INSRMRO', 'INSRMRT', 'INSTAXID', 'RSI', 'SC', 'SI'] from meta as it is not present in specified info cols
Dropping all format fields
Preprocessing GRIDSS callset...
Splitting CIPOS values...
Dropping INFO_CIPOS column...
Returning processed callset...
{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 15, 'chr17': 16, 'chr18': 17, 'chr19': 18, 'chr20': 19, 'chr21': 20, 'chr22': 21, 'chrX': 22, 'chrY': 23, 'chrM': 24}


In [68]:
if 'gnomad' in to_process_list:
    gnomad_vcf = VariantFile(callsets['gnomad']['path'])
    gnomad_df, gnomad_meta = vcfp.parse_vcf_to_df(vcf_file=gnomad_vcf, caller_params=callsets['gnomad']['params'])
    gnomad_df = caller_specific.gnomad_preprocess(gnomad_df, caller_params=callsets['gnomad']['params'], verbose=True)
    gnomad_bnd_df = caller_specific.gnomad_event_to_bnd(gnomad_df)
    gnomad_bp_df = bnds.bnd_to_bp(gnomad_bnd_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    gnomad_bp_df.to_csv(preprocessed_savedir / 'gnomad_bps.csv')
    del gnomad_df, gnomad_meta, gnomad_bnd_df, gnomad_bp_df


[E::idx_find_and_load] Could not retrieve index file for 'data/gnomad.v4.1.sv.sites.vcf.gz'


Dropping ['MULTIALLELIC', 'ALGORITHMS', 'BOTHSIDES_SUPPORT', 'CPX_TYPE', 'LOW_CONFIDENCE_REPETITIVE_LARGE_DUP', 'MEMBERS', 'NCR', 'OUTLIER_SAMPLE_ENRICHED_LENIENT', 'PAR', 'PCRMINUS_NCR', 'PCRPLUS_NCR', 'PESR_GT_OVERDISPERSION', 'PREDICTED_BREAKEND_EXONIC', 'PREDICTED_COPY_GAIN', 'PREDICTED_DUP_PARTIAL', 'PREDICTED_INTERGENIC', 'PREDICTED_INTRAGENIC_EXON_DUP', 'PREDICTED_INTRONIC', 'PREDICTED_INV_SPAN', 'PREDICTED_LOF', 'PREDICTED_MSV_EXON_OVERLAP', 'PREDICTED_NEAREST_TSS', 'PREDICTED_NONCODING_BREAKPOINT', 'PREDICTED_NONCODING_SPAN', 'PREDICTED_PARTIAL_DISPERSED_DUP', 'PREDICTED_PARTIAL_EXON_DUP', 'PREDICTED_PROMOTER', 'PREDICTED_TSS_DUP', 'PREDICTED_UTR', 'RESOLVED_POSTHOC', 'SOURCE', 'UNRESOLVED_TYPE', 'AN', 'AC', 'AF', 'N_BI_GENOS', 'N_HOMREF', 'N_HET', 'N_HOMALT', 'FREQ_HOMREF', 'FREQ_HET', 'FREQ_HOMALT', 'CN_NUMBER', 'CN_COUNT', 'CN_STATUS', 'CN_FREQ', 'CN_NONREF_COUNT', 'CN_NONREF_FREQ', 'AN_afr', 'AC_afr', 'AF_afr', 'N_BI_GENOS_afr', 'N_HOMREF_afr', 'N_HET_afr', 'N_HOMALT_afr',

In [69]:
if 'manta_diploid' in to_process_list:
    manta_diploid_vcf = VariantFile(callsets['manta_diploid']['path'])
    manta_diploid_df, manta_diploid_meta = vcfp.parse_vcf_to_df(vcf_file=manta_diploid_vcf, caller_params=callsets['manta_diploid']['params'])
    manta_bnd_only_df = manta_diploid_df[manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_evt_only_df = manta_diploid_df[~manta_diploid_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_bnd_only_df, caller_params=callsets['manta_diploid']['params'])
    manta_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_evt_only_df, caller_params=callsets['manta_diploid']['params'])
    manta_evt_bnd_df = evts.event_to_bnd(manta_evt_only_df)
    manta_diploid_df = pd.concat([manta_bnd_only_df, manta_evt_bnd_df], axis=0)
    manta_diploid_bp_df = bnds.bnd_to_bp(manta_diploid_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_diploid_bp_df.to_csv(preprocessed_savedir / 'manta_diploid_bps.csv')
    del manta_diploid_df, manta_diploid_meta, manta_bnd_only_df, manta_evt_only_df, manta_evt_bnd_df, manta_diploid_bp_df

Dropping ['END', 'CIGAR', 'HOMSEQ', 'SVINSLEN', 'SVINSSEQ', 'LEFT_SVINSSEQ', 'RIGHT_SVINSSEQ', 'MATE_BND_DEPTH', 'JUNCTION_QUAL'] from meta as it is not present in specified info cols
Dropping ['Ploidy', 'MaxDepth', 'MaxMQ0Frac', 'NoPairSupport', 'MinQUAL', 'SampleFT', 'MinGQ', 'HomRef'] from meta as it is not present in specified filter cols
Dropping ['GT', 'PL'] from meta as it is not present in specified format cols
{'CHROM': 9749, 'POS': 9749, 'ID': 9749, 'REF': 9749, 'ALT': 9749, 'QUAL': 9749}
{'FILTER_PASS': 9749}
{'INFO_SVTYPE': 9749, 'INFO_CIEND': 9749, 'INFO_SVLEN': 9749, 'INFO_CIPOS': 9749, 'INFO_IMPRECISE': 9749, 'INFO_BND_DEPTH': 9749, 'INFO_HOMLEN': 9749, 'INFO_POS2': 9749, 'INFO_EVENT': 9749, 'INFO_MATEID': 9749}
{'SAMPLE_FT': 9749, 'SAMPLE_GQ': 9749, 'SAMPLE_PR': 9749, 'SAMPLE_SR': 9749}
{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 

In [59]:
manta_diploid_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER_PASS,INFO_SVTYPE,INFO_SVLEN,INFO_IMPRECISE,INFO_BND_DEPTH,INFO_HOMLEN,INFO_POS2,INFO_EVENT,INFO_MATEID,SAMPLE_FT,SAMPLE_GQ,INFO_CIPOS_l,INFO_CIPOS_r,INFO_CHR2,SAMPLE_SR_REF,SAMPLE_SR_ALT,SAMPLE_PR_REF,SAMPLE_PR_ALT
5,chr1,1901212,MantaBND:1:19526:19527:0:0:0:1,A,A]chr17:39956249],749.0,1,BND,0,False,78.0,2.0,None,MantaBND:1:19526:19527:0:0:0:1,"(MantaBND:1:19526:19527:0:0:0:0,)",PASS,749,0,2,chr1,53.0,20.0,39,4
31,chr1,5829834,MantaBND:1:4135:4136:0:0:0:0,T,T]chr19:48669149],25.0,1,BND,0,True,146.0,NaN,None,MantaBND:1:4135:4136:0:0:0:0,"(MantaBND:1:4135:4136:0:0:0:1,)",PASS,25,-236,237,chr1,0.0,0.0,11,7
42,chr1,9061384,MantaBND:2387:0:1:0:0:0:1,C,C]chr14:93246833],999.0,1,BND,0,False,59.0,2.0,None,MantaBND:2387:0:1:0:0:0:1,"(MantaBND:2387:0:1:0:0:0:0,)",PASS,999,0,2,chr1,54.0,16.0,55,32
43,chr1,9061390,MantaBND:2387:0:2:0:0:0:0,C,[chr14:93246140[C,876.0,1,BND,0,False,59.0,4.0,None,MantaBND:2387:0:2:0:0:0:0,"(MantaBND:2387:0:2:0:0:0:1,)",PASS,876,0,4,chr1,48.0,17.0,49,19
50,chr1,11431711,MantaBND:1:13984:13987:1:2:0:1,T,T]chr8:19667099],700.0,1,BND,0,False,59.0,59.0,None,MantaBND:1:13984:13987:1:2:0:1,"(MantaBND:1:13984:13987:1:2:0:0,)",PASS,700,0,59,chr1,9.0,9.0,30,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9701,chrX,145990559,MantaINS:1035502:0:0:0:5:0_bp2,T,]chrX:145990559]N,999.0,1,INS,0,False,NaN,1.0,None,MantaINS:1035502:0:0:0:5:0,MantaINS:1035502:0:0:0:5:0_bp1,PASS,473,0,1,NaN,18.0,31.0,26,12
9702,chrX,146392127,MantaINS:13706:659:659:0:0:0_bp2,A,]chrX:146392127]N,999.0,1,INS,160,False,NaN,23.0,None,MantaINS:13706:659:659:0:0:0,MantaINS:13706:659:659:0:0:0_bp1,PASS,165,0,23,NaN,0.0,45.0,2,23
9715,chrX,151743042,MantaINS:493095:6:6:0:0:0_bp2,T,]chrX:151743042]N,946.0,1,INS,60,False,NaN,8.0,None,MantaINS:493095:6:6:0:0:0,MantaINS:493095:6:6:0:0:0_bp1,PASS,61,0,8,NaN,0.0,24.0,0,0
9717,chrX,152833247,MantaINS:1:9918:9918:0:0:0_bp2,C,]chrX:152833247]N,867.0,1,INS,319,False,NaN,15.0,None,MantaINS:1:9918:9918:0:0:0,MantaINS:1:9918:9918:0:0:0_bp1,PASS,409,0,15,NaN,16.0,12.0,21,27


In [58]:
manta_diploid_bp_df

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF,ALT_bnd1,QUAL,FILTER_PASS,INFO_SVTYPE,INFO_SVLEN,INFO_IMPRECISE,INFO_BND_DEPTH_bnd1,INFO_HOMLEN_bnd1,INFO_POS2_bnd1,INFO_EVENT,INFO_MATEID_bnd1,SAMPLE_FT,SAMPLE_GQ,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,INFO_CHR2_bnd1,SAMPLE_SR_REF,SAMPLE_SR_ALT,SAMPLE_PR_REF,SAMPLE_PR_ALT,CHROM_bnd2,POS_bnd2,ID_bnd2,ALT_bnd2,INFO_BND_DEPTH_bnd2,INFO_HOMLEN_bnd2,INFO_POS2_bnd2,INFO_MATEID_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,INFO_CHR2_bnd2
0,chr10,55942264,MantaDEL:100065:0:0:0:0:0_bp1,TGACTTATTCTAATTTCATGGTTTTTTTTTTTTTTTTTTTTTTTTT...,N[chr10:55942605[,458.0,1,DEL,340,False,NaN,22.0,None,MantaDEL:100065:0:0:0:0:0,MantaDEL:100065:0:0:0:0:0_bp2,PASS,97,0,22,NaN,8.0,11.0,11,22,chr10,55942606,MantaDEL:100065:0:0:0:0:0_bp2,]chr10:55942605]N,NaN,22.0,None,MantaDEL:100065:0:0:0:0:0_bp1,0,22,NaN
1,chrX,44552497,MantaDEL:1002259:0:0:0:0:0_bp1,GGATTACAGGCGTGAGCCACCGCACCGGGCTTAATTTTTGTATTTT...,N[chrX:44552635[,142.0,1,DEL,137,False,NaN,21.0,None,MantaDEL:1002259:0:0:0:0:0,MantaDEL:1002259:0:0:0:0:0_bp2,PASS,142,0,21,NaN,25.0,6.0,20,12,chrX,44552636,MantaDEL:1002259:0:0:0:0:0_bp2,]chrX:44552635]N,NaN,21.0,None,MantaDEL:1002259:0:0:0:0:0_bp1,0,21,NaN
2,chrX,47233762,MantaDEL:1003207:0:0:1:0:0_bp1,CTGAGGGGCGTGCTCTGAGGCCGAGGGACGACAGTGGAGGGGAGGT...,N[chrX:47233816[,810.0,1,DEL,53,False,NaN,5.0,None,MantaDEL:1003207:0:0:1:0:0,MantaDEL:1003207:0:0:1:0:0_bp2,PASS,370,0,5,NaN,27.0,21.0,8,0,chrX,47233817,MantaDEL:1003207:0:0:1:0:0_bp2,]chrX:47233816]N,NaN,5.0,None,MantaDEL:1003207:0:0:1:0:0_bp1,0,5,NaN
3,chrX,48017196,MantaDEL:1003436:0:1:0:0:0_bp1,A,N[chrX:48139341[,201.0,1,DEL,122144,False,NaN,6.0,None,MantaDEL:1003436:0:1:0:0:0,MantaDEL:1003436:0:1:0:0:0_bp2,PASS,201,0,6,NaN,28.0,7.0,40,5,chrX,48139342,MantaDEL:1003436:0:1:0:0:0_bp2,]chrX:48139341]N,NaN,6.0,None,MantaDEL:1003436:0:1:0:0:0_bp1,0,6,NaN
4,chrX,48268164,MantaDEL:1003508:0:0:0:0:0_bp1,GGGTCTCACCACCCAAGGGCCTGGGGGCTGGGGTGGGGACGATGAT...,N[chrX:48268327[,999.0,1,DEL,162,False,NaN,2.0,None,MantaDEL:1003508:0:0:0:0:0,MantaDEL:1003508:0:0:0:0:0_bp2,PASS,89,0,2,NaN,0.0,32.0,0,34,chrX,48268328,MantaDEL:1003508:0:0:0:0:0_bp2,]chrX:48268327]N,NaN,2.0,None,MantaDEL:1003508:0:0:0:0:0_bp1,0,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8058,chr10,55002598,MantaINS:99614:0:0:0:0:0_bp1,A,N[chr10:55002599[,999.0,1,INS,63,False,NaN,2.0,None,MantaINS:99614:0:0:0:0:0,MantaINS:99614:0:0:0:0:0_bp2,PASS,39,0,2,NaN,11.0,40.0,0,0,chr10,55002599,MantaINS:99614:0:0:0:0:0_bp2,]chr10:55002599]N,NaN,2.0,None,MantaINS:99614:0:0:0:0:0_bp1,0,2,NaN
8059,chrX,30115713,MantaINS:997119:0:0:0:0:0_bp1,T,N[chrX:30115714[,999.0,1,INS,121,False,NaN,NaN,None,MantaINS:997119:0:0:0:0:0,MantaINS:997119:0:0:0:0:0_bp2,PASS,184,0,0,NaN,0.0,64.0,3,9,chrX,30115714,MantaINS:997119:0:0:0:0:0_bp2,]chrX:30115714]N,NaN,NaN,None,MantaINS:997119:0:0:0:0:0_bp1,0,0,NaN
8060,chrX,30794753,MantaINS:997407:0:0:0:0:0_bp1,T,N[chrX:30794754[,620.0,1,INS,96,False,NaN,38.0,None,MantaINS:997407:0:0:0:0:0,MantaINS:997407:0:0:0:0:0_bp2,PASS,42,0,38,NaN,0.0,19.0,0,0,chrX,30794754,MantaINS:997407:0:0:0:0:0_bp2,]chrX:30794754]N,NaN,38.0,None,MantaINS:997407:0:0:0:0:0_bp1,0,38,NaN
8061,chrX,30793441,MantaINS:997443:0:0:0:0:0_bp1,T,N[chrX:30793442[,999.0,1,INS,78,False,NaN,20.0,None,MantaINS:997443:0:0:0:0:0,MantaINS:997443:0:0:0:0:0_bp2,PASS,121,0,20,NaN,0.0,44.0,0,0,chrX,30793442,MantaINS:997443:0:0:0:0:0_bp2,]chrX:30793442]N,NaN,20.0,None,MantaINS:997443:0:0:0:0:0_bp1,0,20,NaN


In [55]:
manta_evt_only_df['CHROM'].dtype

dtype('O')

In [ ]:
if 'manta_candidate' in to_process_list:
    manta_candidate_vcf = VariantFile(callsets['manta_candidate']['path'])
    manta_candidate_df, manta_candidate_meta = vcfp.parse_vcf_to_df(vcf_file=manta_candidate_vcf, field_info=callsets['manta_candidate']['params'])
    manta_candidate_bnd_only_df = manta_candidate_df[manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_evt_only_df = manta_candidate_df[~manta_candidate_df['INFO_SVTYPE'].isin(['BND', 'INV'])]
    manta_candidate_bnd_only_df = caller_specific.manta_preprocess_bnd_only(manta_candidate_bnd_only_df)
    manta_candidate_evt_only_df = caller_specific.manta_preprocess_evts_only(manta_candidate_evt_only_df)
    manta_candidate_evt_bnd_df = evts.event_to_bnd(manta_candidate_evt_only_df)
    manta_candidate_df = pd.concat([manta_candidate_bnd_only_df, manta_candidate_evt_bnd_df], axis=0)
    manta_candidate_bp_df = bnds.bnd_to_bp(manta_candidate_df, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
    manta_candidate_bp_df.to_csv(preprocessed_savedir / 'manta_candidate_bps.csv')
    del manta_candidate_df, manta_candidate_meta, manta_candidate_bnd_only_df, manta_candidate_evt_only_df, manta_candidate_evt_bnd_df, manta_candidate_bp_df